# 1.2.2 Fitting Models to Curves

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.generators import Math2DGenerator
from src.noise import AdaptiveGaussianNoise
from src.shadows import BandShadow
from src.splitter import GapSplitter
from src.orchestrator import ExperimentOrchestrator
from src.visualization import plot_experiment_results
from src.models import (
    BaseModel,
    PolynomialRegressionModel, 
    DecisionTreeModel, 
    RandomForestModel,
    SVRModel
)

### 1. Kurben Definizioa

In [ ]:
curves = {
    "Exponential": lambda x: np.exp(x * 0.25) + 5,
    "Step (Heaviside)": lambda x: np.where(x > 10, 20, 5),
    "Dampened Sine": lambda x: 10 * np.exp(-0.15 * x) * np.sin(x) + 15,
    "Logistic": lambda x: 20 / (1 + np.exp(-(x - 10))) + 2,
    "Chirp Signal": lambda x: 5 * np.sin(0.1 * x**2) + 10
}

shadow = BandShadow(axis='x', min_val=7, max_val=13)

models_config = [
    (lambda: SVRModel(kernel='rbf', C=100, epsilon=0.1), "SVR (RBF)"),
    (lambda: PolynomialRegressionModel(degree=2), "Poly Degree 2"),
    (lambda: PolynomialRegressionModel(degree=6), "Poly Degree 6"),
    (lambda: DecisionTreeModel(max_depth=None), "DecisionTree"),
    (lambda: RandomForestModel(n_estimators=100), "RandomForest")
]

### 2. Esperimentua Exekutatu

In [ ]:
results_list = []
artifacts_storage = {}

n_samples = 500
x_range = (0, 20)
noise = AdaptiveGaussianNoise(noise_ratio=0.1, seed=42)
splitter = GapSplitter(test_size=0.2, random_state=42)

print("Running Curve Zoo Benchmark...")

for curve_name, func in curves.items():
    print(f"Processing: {curve_name}...")
    gen = Math2DGenerator(func, seed=42)
    
    for model_factory, model_name in models_config:
        model = model_factory()
        
        orchestrator = ExperimentOrchestrator(gen, splitter, model)
        res = orchestrator.run(n_samples, x_range, noise, shadow)
        
        exp_id = f"{curve_name} | {model_name}"
        
        results_list.append({
            "ID": exp_id,
            "Curve": curve_name,
            "Model": model_name,
            "MSE Shadow": res.metrics.mse_test_shadow,
            "MSE Visible": res.metrics.mse_test_visible
        })
        
        artifacts_storage[exp_id] = res

print("Done!")

### 3. Irabazleen eta Galtzaileen Analisia

In [ ]:
df_results = pd.DataFrame(results_list)

# 1. Bilatu onenak (Minimoak)
best_indices = df_results.groupby("Curve")["MSE Shadow"].idxmin()
best_models = df_results.loc[best_indices][['Curve', 'Model', 'MSE Shadow']].rename(columns={'Model': 'Best Model', 'MSE Shadow': 'Best MSE'})

# 2. Bilatu txarrenak (Maximoak)
worst_indices = df_results.groupby("Curve")["MSE Shadow"].idxmax()
worst_models = df_results.loc[worst_indices][['Curve', 'Model', 'MSE Shadow']].rename(columns={'Model': 'Worst Model', 'MSE Shadow': 'Worst MSE'})

# 3. Batu bi taulak
comparison_df = pd.merge(best_models, worst_models, on='Curve')

print("--- PERFORMANCE SUMMARY (Best vs Worst per Curve) ---")
display(comparison_df.sort_values(by='Best MSE'))

### 4. Bistaraketa

In [ ]:
def plot_curve_results(curve_name):
    """Pinta los resultados del Mejor y Peor modelo para una curva dada"""
    curve_data = df_results[df_results['Curve'] == curve_name]
    
    best_row = curve_data.loc[curve_data['MSE Shadow'].idxmin()]
    worst_row = curve_data.loc[curve_data['MSE Shadow'].idxmax()]
    
    print(f"\n>>> ANALISIS: {curve_name}")
    print(f"WINNER: {best_row['Model']} (MSE: {best_row['MSE Shadow']:.4f})")
    print(f"LOSER:  {worst_row['Model']} (MSE: {worst_row['MSE Shadow']:.4f})")
    
    res_best = artifacts_storage[best_row['ID']]
    plot_experiment_results(res_best.split_data, res_best.predictions, 
                            title=f"WINNER: {curve_name} - {best_row['Model']}")
    
    res_worst = artifacts_storage[worst_row['ID']]
    plot_experiment_results(res_worst.split_data, res_worst.predictions, 
                            title=f"LOSER: {curve_name} - {worst_row['Model']}")

In [ ]:
plot_curve_results("Step (Heaviside)")

In [ ]:
plot_curve_results("Chirp Signal")

In [ ]:
plot_curve_results("Dampened Sine")

In [ ]:
plot_curve_results("Exponential")

In [ ]:
plot_curve_results("Logistic")